# 04 — Statistical Analysis

**Owner:** Analytics Lead
**Input:** `data/processed/online_retail_II_clean.csv`
**Output:** Formal test results — every test tied back to a business sub-question
**Rubric link:** Analysis Depth (25 marks — highest weight)

## Ground rule from the rubric
> "Strong statistical analysis — not just counts and averages"

Each section below:
1. States the business question
2. States the null/alternative hypothesis
3. Runs the test with the appropriate method
4. Reports statistic, p-value, effect size, and (critically) **what the result means for the business**

## Tests planned

| # | Business question | Method |
|---|---|---|
| 1 | Is AOV significantly different between registered and guest checkouts? | Welch's t-test |
| 2 | Is return rate independent of product category? | Chi-square test of independence |
| 3 | Do different countries have different monthly revenue distributions? | One-way ANOVA |
| 4 | Is there a month-of-year effect on revenue controlling for customer count? | Linear regression with month dummies |
| 5 | Does first-order basket size predict 12-month customer value? | Pearson correlation + simple linear regression |


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/processed/online_retail_II_clean.csv", parse_dates=["invoicedate"])
df = df[df["is_product"]]
print(df.shape)

## Test 1 — AOV: registered vs guest

**H0:** Mean AOV is equal across registered and guest checkouts.
**H1:** Means differ.
**Method:** Welch's t-test (unequal variances, robust to large-n).
**Why not Student's t:** Variances likely unequal — registered customers have different shopping behavior.

In [ ]:
invoices = (df[~df["is_return"]]
            .groupby(["invoice","is_registered"])["line_revenue"].sum()
            .reset_index())
reg = invoices[invoices["is_registered"]]["line_revenue"]
gst = invoices[~invoices["is_registered"]]["line_revenue"]

t_stat, p_val = stats.ttest_ind(reg, gst, equal_var=False)
# Cohen's d for effect size
pooled_sd = np.sqrt((reg.var()+gst.var())/2)
cohens_d = (reg.mean() - gst.mean()) / pooled_sd
print(f"Registered mean AOV: £{reg.mean():.2f} (n={len(reg):,})")
print(f"Guest      mean AOV: £{gst.mean():.2f} (n={len(gst):,})")
print(f"t = {t_stat:.2f}, p = {p_val:.3g}, Cohen's d = {cohens_d:.3f}")

> **Business interpretation:** *(e.g., 'Registered customers spend 42% more per order — worth the CRM investment')*

## Test 2 — Return rate by product category

**H0:** Return rate is independent of product category hint.
**Method:** Chi-square test of independence on category × is_return contingency table.

In [ ]:
cont = pd.crosstab(df["product_category_hint"], df["is_return"])
chi2, p, dof, expected = stats.chi2_contingency(cont)

# Cramér's V for effect size
n = cont.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(cont.shape) - 1)))
print(cont)
print(f"\nchi2 = {chi2:.2f}, dof = {dof}, p = {p:.3g}, Cramér's V = {cramers_v:.3f}")

> **Business interpretation:** *(which categories are the worst offenders? actionable — can target supplier conversations)*

## Test 3 — Country-level monthly revenue

**H0:** Mean monthly revenue is equal across top-N countries.
**Method:** One-way ANOVA on top-5 countries (UK + 4 largest). Limited to keep groups balanced.

In [ ]:
top_countries = df.groupby("country_clean")["line_revenue"].sum().nlargest(5).index.tolist()
monthly = (df[df["country_clean"].isin(top_countries)]
           .groupby(["country_clean","invoice_year_month"])["line_revenue"].sum()
           .reset_index())

groups = [g["line_revenue"].values for _, g in monthly.groupby("country_clean")]
f_stat, p_val = stats.f_oneway(*groups)
print(f"Countries: {top_countries}")
print(f"F = {f_stat:.2f}, p = {p_val:.3g}")

# Follow-up: which pairs actually differ
from statsmodels.stats.multicomp import pairwise_tukeyhsd
tukey = pairwise_tukeyhsd(monthly["line_revenue"], monthly["country_clean"])
print(tukey)

> **Business interpretation:** *(comment on whether differences are driven by UK alone or there are real int'l tiers worth targeting)*

## Test 4 — Month-of-year effect on revenue

**H0:** Monthly revenue has no seasonal effect once we account for active-customer count.
**Method:** OLS regression — revenue ~ month + active_customers.

In [ ]:
monthly_agg = (df[df["is_registered"]]
               .groupby("invoice_year_month")
               .agg(revenue=("line_revenue","sum"),
                    active_customers=("customer_id","nunique"))
               .reset_index())
monthly_agg["month_num"] = pd.to_datetime(monthly_agg["invoice_year_month"]).dt.month
monthly_agg["month_cat"] = "M" + monthly_agg["month_num"].astype(str).str.zfill(2)

model = smf.ols("revenue ~ C(month_cat) + active_customers", data=monthly_agg).fit()
print(model.summary())

> **Business interpretation:** *(which months significantly over/under-index after controlling for customer count? this is the inventory-planning insight)*

## Test 5 — First-order basket size → 12-month customer value

**H0:** No linear relationship between first-order revenue and 12-month customer revenue.
**Method:** Pearson correlation + simple linear regression.

In [ ]:
reg_df = df[df["is_registered"] & ~df["is_return"]].copy()
# First invoice per customer
first_inv = reg_df.sort_values("invoicedate").groupby("customer_id").first().reset_index()
first_rev = reg_df.merge(first_inv[["customer_id","invoice"]], on=["customer_id","invoice"])
first_order_rev = first_rev.groupby("customer_id")["line_revenue"].sum().rename("first_order_rev")

# 12-month revenue from first purchase
reg_df["first_date"] = reg_df.groupby("customer_id")["invoicedate"].transform("min")
reg_df["days_since_first"] = (reg_df["invoicedate"] - reg_df["first_date"]).dt.days
twelve_mo = (reg_df[reg_df["days_since_first"] <= 365]
             .groupby("customer_id")["line_revenue"].sum().rename("ltv_12m"))

ltv = pd.concat([first_order_rev, twelve_mo], axis=1).dropna()

r, p = stats.pearsonr(ltv["first_order_rev"], ltv["ltv_12m"])
print(f"Pearson r = {r:.3f}, p = {p:.3g}")

# Linear regression
X = sm.add_constant(ltv["first_order_rev"])
model = sm.OLS(ltv["ltv_12m"], X).fit()
print(model.summary())

> **Business interpretation:** *(does a bigger first order predict higher LTV? if yes → invest in first-order upsell)*

## Summary table

| Test | Statistic | p-value | Effect size | Decision |
|---|---|---|---|---|
| 1 — AOV reg vs guest | *fill* | *fill* | *fill* | *fill* |
| 2 — Return × category | *fill* | *fill* | *fill* | *fill* |
| 3 — Country ANOVA | *fill* | *fill* | *fill* | *fill* |
| 4 — Month regression | *fill* | *fill* | *fill* | *fill* |
| 5 — First-order → LTV | *fill* | *fill* | *fill* | *fill* |

These five rows become the "Statistical Analysis Results" section of the final report.
